In [1]:
from langchain.schema.runnable import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
import dotenv
dotenv.load_dotenv()

True

### PromptTemplate 에 기사내용

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

prompt = PromptTemplate.from_template("""
다음의 context를 읽고, 질문에 답해줘.
context: {context}
질문: {question}
""")

chain = prompt | llm

response = chain.invoke({
    'context': '''
다음 기사를 근거로 질문에 답하세요.
-----
제21대 대한민국 대통령에 이재명 더불어민주당 후보가 당선됐다.

초유의 비상계엄 사태와 윤석열 전 대통령 파면 속에 치러진 조기 대선에서 민심이 정권 교체를 선택한 것이다. 전국 최종 투표율은 79.4%를 기록, 총 3524만416명이 투표에 참여했다.

중앙선거관리위원회에 따르면 4일 오전 5시 10분 개표율 100%를 기준으로 기호 1번 더불어민주당 이재명 후보가 1728만7513표로 전체 49.42%를 득표했다. 1439만5639표를 얻은 기호 2번 국민의힘 김문수 후보(41.15%)를 8.27%p차로 앞서며 당선을 확정 지었다.

앞서 방송 3사(KBS·MBC·SBS) 출구조사에서는 이재명 후보가 51.7%를 득표해 과반을 넘길 것이라 예측됐지만, 최종적으로 과반을 하지는 못했다.

이재명 후보는 제21대 대통령 당선이 확실시된 4일 오전 서울 여의도 국회 앞에 마련된 야외무대에 "민주공화국 대한민국 시민 여러분께 진심으로 감사드린다"며 "여러분들이 제게 기대하시고 맡긴 그 사명을 한순간도 잊지 않고 한 치의 어긋남도 없이 반드시, 확실히 이행하겠다"고 말했다.
-----''',
    'question': "한국의 대통령은?"
})
print(response.content)

이재명


### Hub에서 pull 한 prompt에 기사내용

In [5]:
from langchain import hub
prompt = hub.pull('rlm/rag-prompt')
print(prompt)

chain = prompt | llm

response = chain.invoke({
    'context': '''다음 기사를 근거로 질문에 답하세요.
-----
제21대 대한민국 대통령에 이재명 더불어민주당 후보가 당선됐다.
초유의 비상계엄 사태와 윤석열 전 대통령 파면 속에 치러진 조기 대선에서 민심이 정권 교체를 선택한 것이다. 전국 최종 투표율은 79.4%를 기록, 총 3524만416명이 투표에 참여했다.
중앙선거관리위원회에 따르면 4일 오전 5시 10분 개표율 100%를 기준으로 기호 1번 더불어민주당 이재명 후보가 1728만7513표로 전체 49.42%를 득표했다. 1439만5639표를 얻은 기호 2번 국민의힘 김문수 후보(41.15%)를 8.27%p차로 앞서며 당선을 확정 지었다.
앞서 방송 3사(KBS·MBC·SBS) 출구조사에서는 이재명 후보가 51.7%를 득표해 과반을 넘길 것이라 예측됐지만, 최종적으로 과반을 하지는 못했다.
이재명 후보는 제21대 대통령 당선이 확실시된 4일 오전 서울 여의도 국회 앞에 마련된 야외무대에 "민주공화국 대한민국 시민 여러분께 진심으로 감사드린다"며 "여러분들이 제게 기대하시고 맡긴 그 사명을 한순간도 잊지 않고 한 치의 어긋남도 없이 반드시, 확실히 이행하겠다"고 말했다.
-----''',
    'question': '한국의 대통령은?'
})
print(response.content)

input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]
기사에 따르면 대한민국의 제21대 대통령은 이재명 더불어민주당 후보입니다. 그는 49.42%의 득표율로 당선되었습니다. 그는 민심이 정권 교체를 선택한 조기 대선에서 당선되었습니다.


### Simple Retriever 이용

In [ ]:
class SimpleRetriever(BaseRetriever):
    docs: list[Document]
    k: int = 5

    def _get_relevant_documents(self, query: str) -> list[Document]:
        return self.docs[:self.k]


document = Document(
    page_content='2025년 6월 3일에 당선된 제 21대 대통령은 더불어민주당 이재명이다.',
    metadata={'source': 'https://example.com'}
)

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

prompt = PromptTemplate.from_template("""
다음의 context를 읽고, 질문에 답해줘.
context: {context}
질문: {question}
""")

retriever = SimpleRetriever(docs=[document], k=1)
context = retriever.invoke('한국의 대통령은?')
# print(context)

chain = prompt | llm

response = chain.invoke({
    'question': '한국의 대통령은?',
    'context': retriever.invoke('한국의 대통령은?')
})
# print(response.content)

chain = {
    'context': retriever,
    'question': RunnablePassthrough()
} | prompt | llm
response = chain.invoke('한국의 대통령은?')
print(response.content)

한국의 대통령은 더불어민주당 이재명입니다.


### Wikipedia Retriever

In [10]:
from langchain.schema.runnable import RunnablePassthrough
from langchain_community.retrievers import WikipediaRetriever
retriever = WikipediaRetriever()

chain = {
    'context': retriever,
    'question': RunnablePassthrough()
} | prompt | llm

context = retriever.invoke('한국의 대통령은?')
print(context)

response = chain.invoke('한국의 대통령은?')
print(response.content)

[Document(metadata={'title': 'Democratic Party (South Korea, 2015)', 'summary': "The Democratic Party of Korea (DPK or DP; Korean: 더불어민주당, lit.\u2009'Together Democratic Party') is a liberal political party in South Korea. The DPK and its rival, the People Power Party (PPP), form the two major political parties of South Korea. It is the ruling party following the victory of Lee Jae Myung at the 2025 presidential election, and has been the largest party in the National Assembly since 2016, controlling a majority since 2020. It was previously the ruling party under Moon Jae-in from 2017 to 2022.\nThe Democratic Party was founded as the New Politics Alliance for Democracy (NPAD; 새정치민주연합) on 26 March 2014 as a merger between the previous Democratic Party and the preparatory committee of the New Political Vision Party (NPVP) led by Ahn Cheol-soo. The party changed its name to the current name on 28 December 2015. In 2022, the Democratic Party, the Open Democratic Party, and New Wave merged 

### Vector Store as Retriever

In [ ]:
from abc import ABC
from typing import Any
# from langchain_core.retrievers import VectorStoreRetriever

from langchain_core.callbacks import CallbackManagerForRetrieverRun


class VectorStoreRetriever(BaseRetriever):
    def __init__(self, vector_store, tags: list[str] = None, **kwargs: Any):
        super().__init__(**kwargs)
        self.vector_store = vector_store
        self.tags = tags or []

    def _get_relevant_documents(self, query, *, run_manager: CallbackManagerForRetrieverRun, **kwargs: Any) -> list[Document]:
        return self.vector_store.similarity_search(query, **kwargs)


class VectorStore(ABC):
    def as_retriever(self, **kwargs: Any) -> VectorStoreRetriever:
        raise VectorStoreRetriever(vector_store=self, **kwargs)

In [29]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
#  embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")

vector_store = InMemoryVectorStore(embeddings)

docs = [
    Document(page_content='2025년 6월 3일에 당선된 제 21대 대통령은 더불어민주당 이재명이다.',
             metadata={'source': 'https://example.com'}),
    Document(page_content='삼성 가우스는 삼성전자의 멀티모달 모델의 생성형 인공지능이다.',
             metadata={'source': 'https://example.com'}),
]
vector_store.add_documents(docs)

similar_docs = vector_store.similarity_search('한국의 대통령은?', k=2)
print(similar_docs)

prompt = PromptTemplate.from_template("""
다음의 context를 읽고, 질문에 답해줘.
context: {context}
질문: {question}
""")

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
chain = {
    'context': vector_store.as_retriever(),
    'question': RunnablePassthrough()
} | prompt | llm

response = chain.invoke('한국의 대통령은?')
print(response.content)

[Document(id='38392d7b-e05b-41c0-b474-f83c7a345a5f', metadata={'source': 'https://example.com'}, page_content='삼성 가우스는 삼성전자의 멀티모달 모델의 생성형 인공지능이다.'), Document(id='3a26a025-7da4-4e72-a405-bf457a4c7188', metadata={'source': 'https://example.com'}, page_content='2025년 6월 3일에 당선된 제 21대 대통령은 더불어민주당 이재명이다.')]
한국의 대통령은 더불어민주당 이재명입니다.


### Web Base Loader

In [36]:
from langchain_google_genai import ChatGoogleGenerativeAI
import bs4
import os
from langchain_community.document_loaders import WebBaseLoader

os.environ["USER_AGENT"] = "Mozilla/5.0"

loader = WebBaseLoader(
    web_path="https://n.news.naver.com/article/001/0015568637",
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer('article')
    )
)

docs = loader.load()
#  print(docs[0].page_content)

llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash')

response = llm.invoke(f'다음 기사를 한 문장으로 요약해줘 : {docs[0].page_content}')
print(response.content)

오픈AI의 최신 모델 GPT-5가 기대 이하의 성능으로 오류와 잘못된 답변을 연발하며 사용자들의 조롱을 받고, 이전 버전으로 되돌리는 소동까지 벌어졌다.


### PyPDFLoader

In [15]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data\\AI 에이전트 동향.pdf", mode='single')
docs = loader.load()

with open("data\\AI 에이전트 동향.txt", "w", encoding="utf-8") as f:
    f.write(docs[0].page_content)

### RecursiveCharacterTextSplitter

In [16]:
with open('data\\AI 에이전트 동향.txt', 'r', encoding='utf-8') as f:
    file = f.read()

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter()

docs = splitter.create_documents([file])

with open('data\\AI 에이전트 동향_splitted.txt', 'w', encoding='utf-8') as f:
    f.write(f'Number of splitted documents: {len(docs)}')
    for doc in docs:
        f.write(f'\n\n---\n\n{doc.page_content}')

### MarkdownHeaderTextSplitter

In [22]:
with open('data\\langchain.md', 'r', encoding='utf-8') as f:
    file = f.read()

from langchain_text_splitters import MarkdownHeaderTextSplitter
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[('#', 'Chapter'), ('##', 'Section')],
    strip_headers=False,
)

docs = splitter.split_text(file)
for i, doc in enumerate(docs):
    print(i, 'th\n', doc.page_content[:100])

0 th
 # Introduction  
**LangChain** is a framework for developing applications powered by large language 
1 th
 ## Architecture  
The LangChain framework consists of multiple open-source libraries. Read more in t
2 th
 ## Guides  
### [Tutorials](/docs/tutorials)  
If you're looking to build something specific or are 
3 th
 ## Ecosystem  
### [🦜🛠️ LangSmith](https://docs.smith.langchain.com)
Trace and evaluate your languag
4 th
 ## Additional resources  
### [Versions](/docs/versions/v0_3/)
See what changed in v0.3, learn how t


### SemanticChunker

In [31]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# pip install langchain_experimental
splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type='percentile',
    breakpoint_threshold_amount=50,
)

docs = splitter.create_documents([file])
for i, doc in enumerate(docs):
    print(i, 'th\n', doc.page_content[:50])

0 th
 # Introduction

**LangChain** is a framework for d
1 th
 - **Deployment**: Turn your LangGraph applications
2 th
 See the [integrations](/docs/integrations/provider
3 th
 import ChatModelTabs from "@theme/ChatModelTabs";

4 th
 :::

## Architecture

The LangChain framework cons
5 th
 Read more in the
[Architecture](/docs/concepts/arc
6 th
 - **Integration packages** (e.g. `langchain-openai
7 th
 - **`langchain-community`**: Third-party integrati
8 th
 See [LangGraph documentation](https://langchain-ai
9 th
 ## Guides

### [Tutorials](/docs/tutorials)

If yo
10 th
 This is the best place to get started. These are t
11 th
 ### [How-to guides](/docs/how_to)

[Here](/docs/ho
12 th
 These how-to guides don’t cover topics in depth – 
13 th
 ### [Conceptual guide](/docs/concepts)

Introducti
14 th
 For a deeper dive into LangGraph concepts, check o
15 th
 If you're looking to get up and running quickly wi
16 th
 ## Ecosystem

### [🦜🛠️ LangSmith](https://docs.smi
17 th
 ### [🦜🕸️ LangGrap

### LLM Splitter

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
with open('data\\AI 에이전트 동향.txt', 'r', encoding='utf-8') as f:
    file = f.read()

from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template('''
다음의 context를 읽고, 질문에 답해줘.
context: {context}
질문: {question}
''')

llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash')

chain = prompt | llm
# response = chain.invoke(file)
# print(response.content)
print(file)